In [1]:
import sys
# sys.path.append(r"E:\Dai hoc\2526I\dacn\flow-matching")
# sys.path.append(r"E:\Dai hoc\2526I\dacn\flow-matching\demo-code\2d")
import torch
torch.set_default_device("cuda")
import numpy as np
import h5py
from collections import defaultdict, Counter
import numpy as np
from rich import print

from flow_model import HCDFlowResMLP, HCDFlow

In [ ]:
i_1 = torch.tensor([[1, 2, 3], 
                    [3, 4, 5]], dtype=torch.float64)
i_2 = torch.tensor([[2, 2, 4],
                    [3, 3, 6]], dtype=torch.float64)

In [55]:
i_1 - i_1.mean(dim=1, keepdim=True)

tensor([[-1.,  0.,  1.],
        [-1.,  0.,  1.]], device='cuda:0', dtype=torch.float64)

In [47]:
(i_2 - i_1).abs().sum(dim=1).max()

tensor(4., device='cuda:0', dtype=torch.float64)

# Load pretrain model

In [ ]:
model_path = "hello"
model = HCDFlowResMLP(noise_dim=174, pep_dim=256, time_dim=128, charge_dim=128, num_blocks=10)
model.load_state_dict(torch.load(model_path))

model.eval()

# Test on training data

In [ ]:
train_path = r"E:\Dai hoc\2526I\dacn\flow-matching\data\traintest_hcd.hdf5"

In [ ]:
with h5py.File(train_path, "r") as f:
    print("Keys:", list(f.keys()))
    seqs = f["sequence_integer"][:]
    charges_oh = f["precursor_charge_onehot"][:]
    intensities = f["intensities_raw"][:]  

In [ ]:
charges = np.argmax(charges_oh, axis=1) + 1
del charges_oh

In [ ]:
batch = 1024
total_sample = len(seqs)
loss_l1 = 0
pcc = 0
for i in range(0, total_sample, batch):
    end_index = min(i + batch, total_sample)
    seq_tensors = torch.tensor(seqs[i:end_index])
    charge_tensors = torch.tensor(charges[i:end_index])
    intensity_tensors = torch.tensor(intensities[i:end_index])
    
    noise = torch.randn_like(intensity_tensors)
    
    generated_intensities = model.sample(noise, seq_tensors, charge_tensors, step=20)
    
    

# Test on test set